# AURA Free GPU Training - Google Colab
## Fine-tune Qwen2.5-1.5B with Unsloth on Free T4 GPU

**Instructions:**
1. Open this notebook in Google Colab: https://colab.research.google.com/
2. Go to Runtime > Change runtime type > Select **T4 GPU** > Save
3. Click **Connect** in the top right
4. Run all cells (Runtime > Run all)

**What this does:**
- Installs Unsloth for 2x faster training with 70% less VRAM
- Loads Qwen2.5-1.5B in 4-bit quantization
- Downloads the pre-merged AURA training data (12,498 samples) from HuggingFace Hub
- Fine-tunes using QLoRA on free T4 GPU
- Saves the fine-tuned model and pushes to Hugging Face Hub

In [ ]:
# @title 1. Install Unsloth and Dependencies (2-3 min)
# @markdown This installs the optimized training library
%%capture
# Unsloth manages its own dependency versions; pin xformers is NOT needed
# and the old ==0.0.29.post3 pin breaks on Colab's PyTorch 2.5+
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install --no-deps unsloth

print("Unsloth installed successfully!")
!nvidia-smi

In [ ]:
# @title 2. Hugging Face Authentication
# @markdown Enter your Hugging Face token to push the model after training
# @markdown and to download the pre-merged training dataset

from google.colab import userdata
from huggingface_hub import login, HfApi

HF_TOKEN = userdata.get('HF_TOKEN')  # @param {type:"string"}

if HF_TOKEN:
    login(token=HF_TOKEN)
    api = HfApi()
    user_info = api.whoami()
    print(f"Logged in as: {user_info['name']}")
else:
    print("No HF_TOKEN provided. Training data download may be rate-limited.")
    print("Add HF_TOKEN in Colab Secrets ( Settings > Secrets > Add a new secret)")

In [ ]:
# @title 3. Training Configuration
# @markdown Adjust these parameters for your training run

from unsloth import FastLanguageModel
import torch

# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # @param {type:"string"}
MAX_SEQ_LENGTH = 2048  # @param {type:"number"}
LOAD_IN_4BIT = True  # @param {type:"boolean"}

# LoRA configuration
LORA_R = 16  # @param {type:"number"}
LORA_ALPHA = 16  # @param {type:"number"}
LORA_DROPOUT = 0.05  # @param {type:"number"}

# Training configuration
BATCH_SIZE = 4  # @param {type:"number"} -- doubled from 2 (12K samples need fewer steps)
GRADIENT_ACCUMULATION_STEPS = 4  # @param {type:"number"}
NUM_EPOCHS = 1  # @param {type:"number"} -- 1 epoch on 12,498 samples is sufficient for QLoRA
LEARNING_RATE = 2e-4  # @param {type:"number"}
WARMUP_STEPS = 10  # @param {type:"number"}

# Dataset configuration — uses the FULL 12,498-sample merged dataset by default
USE_FULL_DATASET = True  # @param {type:"boolean"}
MAX_TRAINING_SAMPLES = 12498  # @param {type:"number"}

print(f"Configuration loaded:")
print(f"  Model: {MODEL_NAME}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  LoRA rank: {LORA_R}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Use full dataset: {USE_FULL_DATASET}")

In [ ]:
# @title 4. Load Model with 4-bit Quantization
# @markdown Loads Qwen2.5-1.5B optimized for T4 GPU

print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (Float16 for T4, Bfloat16 for Ampere+)
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# @title 5. Add LoRA Adapters
# @markdown Configure parameter-efficient fine-tuning

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Critical for memory savings
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

In [ ]:
# @title 6. Download Training Data
# @markdown Downloads the pre-merged AURA training dataset (12,498 samples)
# @markdown from HuggingFace Hub — no manual upload needed.
# @markdown
# @markdown If the dataset repo is not public yet, it falls back to manual upload.

import os
import shutil

os.makedirs("aura_training_data", exist_ok=True)

DATASET_REPO = "raiden456/aura-training-data"  # HF dataset repo
DATASET_FILENAME = "aura_merged_training.jsonl"

downloaded = False
try:
    from huggingface_hub import hf_hub_download
    local_path = hf_hub_download(
        repo_id=DATASET_REPO,
        filename=DATASET_FILENAME,
        repo_type="dataset",
    )
    dest = f"aura_training_data/{DATASET_FILENAME}"
    shutil.copy(local_path, dest)
    with open(dest, "r", encoding="utf-8") as f:
        n = sum(1 for _ in f)
    print(f"Downloaded {DATASET_FILENAME} ({n} samples) from {DATASET_REPO}")
    downloaded = True
except Exception as e:
    print(f"Could not download from HF Hub: {e}")
    print("The dataset repo may not exist yet or you are not authenticated.")

if not downloaded:
    print("\nFalling back to manual upload...")
    print("Upload aura_merged_training.jsonl (12,498 samples) from your local machine.")
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, f"aura_training_data/{filename}")
        print(f"Uploaded: {filename}")

In [ ]:
# @title 7. Prepare Training Data
# @markdown Loads the merged JSONL and formats for Qwen2.5 chat template

import glob
import json
from datasets import Dataset
from unsloth.chat_templates import standardize_sharegpt, get_chat_template

def load_and_merge_jsonl(data_dir="aura_training_data", max_samples=None):
    """Load all JSONL files and merge into a single dataset."""
    all_data = []
    jsonl_files = glob.glob(f"{data_dir}/*.jsonl")

    if not jsonl_files:
        print("No JSONL files found! Please run the data download cell first.")
        return None

    for filepath in jsonl_files:
        print(f"Loading: {os.path.basename(filepath)}")
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    entry = json.loads(line)
                    # Handle Alpaca format: instruction + input + output
                    if "instruction" in entry and "output" in entry:
                        instruction = entry["instruction"]
                        user_content = instruction
                        if entry.get("input", "").strip():
                            user_content = f"{instruction}\n\n{entry['input'].strip()}"
                        all_data.append({
                            "conversations": [
                                {"role": "user", "content": user_content},
                                {"role": "assistant", "content": entry["output"]},
                            ]
                        })
                    # Handle AURA format: text=input, output=response
                    elif "text" in entry and "output" in entry:
                        all_data.append({
                            "conversations": [
                                {"role": "user", "content": entry["text"]},
                                {"role": "assistant", "content": entry["output"]},
                            ]
                        })
                    # Handle shared GPT format directly
                    elif "conversations" in entry:
                        all_data.append(entry)
                except json.JSONDecodeError:
                    continue

                if max_samples and len(all_data) >= max_samples:
                    break

        if max_samples and len(all_data) >= max_samples:
            break

    print(f"\nTotal samples loaded: {len(all_data)}")
    return all_data

# Load data
max_samples = MAX_TRAINING_SAMPLES if not USE_FULL_DATASET else None
raw_data = load_and_merge_jsonl(max_samples=max_samples)

if raw_data:
    # Convert to HuggingFace dataset
    dataset = Dataset.from_list(raw_data)
    print(f"Dataset created with {len(dataset)} samples")

    # Standardize format for Qwen2.5
    dataset = standardize_sharegpt(dataset)

    # Apply Qwen2.5 chat template
    tokenizer = get_chat_template(tokenizer, chat_template="qwen_2.5")

    def formatting_prompts_func(examples):
        texts = [
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False
            )
            for convo in examples["conversations"]
        ]
        return {"text": texts}

    dataset = dataset.map(formatting_prompts_func, batched=True)
    print("Data prepared with Qwen2.5 chat template!")
    print(f"\nSample formatted text:\n{dataset[0]['text'][:500]}...")
else:
    print("ERROR: No training data loaded. Training cannot proceed.")

In [ ]:
# @title 8. Train the Model
# @markdown Fine-tune Qwen2.5-1.5B with QLoRA on your data
# @markdown
# @markdown With 12,498 samples, 1 epoch, batch_size=4, grad_accum=4:
# @markdown effective batch size = 16, ~781 steps, ~10-15 min on T4.

from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Start training
print("Starting training...")
trainer_stats = trainer.train()
print("Training complete!")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# @title 9. Test the Fine-Tuned Model
# @markdown Try out your newly trained model

FastLanguageModel.for_inference(model)

test_prompts = [
    "What is AURA?",
    "How do I create a new AME?",
    "What integrations are supported?",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        min_p=0.1,
    )

    response = tokenizer.batch_decode(outputs)[0]
    print(f"\nUser: {prompt}")
    print(f"AURA: {response.split('assistant')[-1].strip()}")
    print("-" * 50)

In [ ]:
# @title 10. Save and Export Model
# @markdown Save locally and optionally push to Hugging Face Hub

# Save LoRA adapters
model.save_pretrained("aura_finetuned_lora")
tokenizer.save_pretrained("aura_finetuned_lora")
print("LoRA adapters saved to: aura_finetuned_lora")

# Merge and save full model
print("\nMerging LoRA with base model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("aura_finetuned_merged")
tokenizer.save_pretrained("aura_finetuned_merged")
print("Merged model saved to: aura_finetuned_merged")

# Export to GGUF for Ollama/LM Studio
print("\nExporting to GGUF format...")
model.save_pretrained_gguf(
    "aura_finetuned_gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Good balance of size and quality
)
print("GGUF model saved to: aura_finetuned_gguf")

# Push to Hugging Face Hub if token provided
if HF_TOKEN:
    print("\nPushing to Hugging Face Hub...")
    try:
        model.push_to_hub("aura_finetuned_lora", token=HF_TOKEN)
        tokenizer.push_to_hub("aura_finetuned_lora", token=HF_TOKEN)
        print("Model pushed to Hugging Face Hub!")
        print("Download later with: from_pretrained('your-username/aura_finetuned_lora')")
    except Exception as e:
        print(f"Failed to push: {e}")
        print("Model saved locally. Push manually with: huggingface-cli upload")
else:
    print("\nNo HF_TOKEN provided. Model saved locally only.")
    print("Download the files from Colab's file browser ( folder icon)")

In [ ]:
# @title 11. Download Fine-Tuned Model
# @markdown Download the model files to your local machine

from google.colab import files
import shutil

# Create a zip file with all outputs
shutil.make_archive("aura_finetuned_model", 'zip', ".", "aura_finetuned_merged")

print("Downloading fine-tuned model...")
files.download("aura_finetuned_model.zip")
print("Download complete!")
print("\nExtract the zip and use with Ollama, llama.cpp, or your local app.")